# FinReflectKG: ทดลองดาวน์โหลดและกรองข้อมูลทีละขั้น

Notebook นี้แยกโค้ดจาก scripts/fetch_ds.py เป็นส่วนย่อย เพื่อให้ดูค่าที่แต่ละส่วนรับเข้าและส่งออกได้

**ก่อนเริ่ม:** เลือก Kernel ของ Python environment ที่มี pyarrow และ huggingface_hub เช่น senior_project_py312 แล้วรันเซลล์จากบนลงล่าง

เดโมนี้เลือก shard ลำดับที่ 10 (index 9) ซึ่งตรวจพบแถวของ ADI ใน snapshot ปัจจุบัน และบันทึกผลไว้ในโฟลเดอร์ cache แยกจากไฟล์ data/finreflectkg_sox.parquet หลัก


## คำที่ใช้ใน Notebook นี้

- **Parquet** คือรูปแบบไฟล์เก็บข้อมูลตาราง
- **Shard** คือไฟล์ส่วนหนึ่งของชุดข้อมูลขนาดใหญ่ เช่น train-00009-of-00103.parquet
- หนึ่งแถวใน FinReflectKG คือข้อมูล **triple** หนึ่งรายการ พร้อม metadata และข้อความบริบท chunk_text แถวหลายแถวอาจมี chunk_text เดียวกันได้
- ตัวแปร local_path เป็นตำแหน่งไฟล์บนเครื่อง ส่วน table เป็นตารางที่อ่านและกรองแล้ว


## 1. นำเข้าเครื่องมือ

PyArrow ใช้อ่านและเขียน Parquet ส่วน Hugging Face Hub ใช้ดูรายชื่อไฟล์และดาวน์โหลด shard


In [1]:
import time
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download

print('pyarrow:', pa.__version__)
print('เครื่องมือพร้อมใช้งาน')


pyarrow: 25.0.1
เครื่องมือพร้อมใช้งาน


/home/kantinan/miniconda3/envs/senior_project_py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. ตั้งค่าที่ใช้

รายชื่อ ticker และชื่อ dataset ใช้ตามสคริปต์เดิม ส่วน cache และไฟล์ผลลัพธ์ของ Notebook แยก path ออกจากผลลัพธ์หลัก เพื่อไม่เขียนทับไฟล์เดิม


In [6]:
SOX_TICKERS = [
    'NVDA', 'AMD', 'INTC', 'AVGO', 'TXN', 'AMAT',
    'QCOM', 'ADI', 'MU', 'MRVL', 'MCHP', 'KLAC', 'LRCX',
]
REPO_ID = 'domyn/FinReflectKG'

# ไฟล์เดโมแยกจาก data/finreflectkg_sox.parquet หลัก
DEMO_DIR = Path.home() / '.cache' / 'semigraph' / 'fetch_ds_notebook_demo'
CACHE_DIR = DEMO_DIR / 'cache'
OUTPUT_PATH = DEMO_DIR / 'finreflectkg_sox_demo.parquet'

# index เริ่มนับจาก 0 ดังนั้น 9 คือ shard ลำดับที่ 10
DEMO_SHARD_INDEX = 9

print('dataset:', REPO_ID)
print('จำนวน ticker:', len(SOX_TICKERS))
print('cache path:', CACHE_DIR)
print('output path:', OUTPUT_PATH)
print('เลือก shard index:', DEMO_SHARD_INDEX)


dataset: domyn/FinReflectKG
จำนวน ticker: 13
cache path: /home/kantinan/.cache/semigraph/fetch_ds_notebook_demo/cache
output path: /home/kantinan/.cache/semigraph/fetch_ds_notebook_demo/finreflectkg_sox_demo.parquet
เลือก shard index: 9


## 3. ขอรายชื่อ shard จาก Hugging Face

ส่วนนี้คืนค่าเป็น list ของชื่อไฟล์ แล้วเลือกเฉพาะไฟล์ในโฟลเดอร์ data ที่ลงท้ายด้วย .parquet


In [7]:
api = HfApi()
repo_files = api.list_repo_files(REPO_ID, repo_type='dataset')
shards = sorted(
    path for path in repo_files
    if path.startswith('data/') and path.endswith('.parquet')
)

print('ชนิดค่าที่ได้จากการกรอง:', type(shards).__name__)
print('จำนวน shard:', len(shards))
print('ตัวอย่าง 3 ไฟล์แรก:', shards[:3])
print('ไฟล์สุดท้าย:', shards[-1] if shards else 'ไม่พบไฟล์')


ชนิดค่าที่ได้จากการกรอง: list
จำนวน shard: 103
ตัวอย่าง 3 ไฟล์แรก: ['data/train-00000-of-00103.parquet', 'data/train-00001-of-00103.parquet', 'data/train-00002-of-00103.parquet']
ไฟล์สุดท้าย: data/train-00102-of-00103.parquet


## 4. เลือก shard สำหรับเดโม

ปกติ --max 1 ในสคริปต์เดิมจะเลือก shard แรก แต่ shard แรกอาจไม่มี ticker ที่เราต้องการ Notebook นี้เลือก index 9 เพื่อให้เห็นตัวอย่างที่มีแถว ADI หาก dataset เปลี่ยนแล้วไม่พบแถว ให้ลองเปลี่ยน DEMO_SHARD_INDEX


In [8]:
if not shards:
    raise RuntimeError('ไม่พบ Parquet shard ใน dataset')
if DEMO_SHARD_INDEX >= len(shards):
    raise IndexError(f'มี shard เพียง {len(shards)} ไฟล์ แต่เลือก index {DEMO_SHARD_INDEX}')

shard_path = shards[DEMO_SHARD_INDEX]
print('ชนิดตัวแปร shard_path:', type(shard_path).__name__)
print('ค่า shard_path:', shard_path)


ชนิดตัวแปร shard_path: str
ค่า shard_path: data/train-00009-of-00103.parquet


## 5. ดาวน์โหลด shard และดูค่า return

download_shard() รับชื่อไฟล์จาก Hugging Face แล้วคืน path ของไฟล์บนเครื่องเป็น str ถ้าดาวน์โหลดไม่สำเร็จ จะลองใหม่สูงสุด 3 ครั้ง โดยเว้น 5, 10 และ 15 วินาที


In [9]:
def download_shard(shard_path: str, retries: int = 3) -> str:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for attempt in range(retries):
        try:
            local = hf_hub_download(
                repo_id=REPO_ID,
                filename=shard_path,
                repo_type='dataset',
                cache_dir=str(CACHE_DIR),
            )
            return local
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f'ลองใหม่ {attempt + 1}/{retries} ใน {wait} วินาที ({e.__class__.__name__})')
            time.sleep(wait)
    raise RuntimeError(f'ดาวน์โหลดไม่สำเร็จ: {shard_path}')


In [10]:
local_path = download_shard(shard_path)
local_file = Path(local_path)

print('ชนิดค่าที่คืนมา:', type(local_path).__name__)
print('ค่า local_path:', local_path)
print('ไฟล์มีอยู่:', local_file.exists())
print('ขนาดไฟล์ (bytes):', local_file.stat().st_size)


ชนิดค่าที่คืนมา: str
ค่า local_path: /home/kantinan/.cache/semigraph/fetch_ds_notebook_demo/cache/datasets--domyn--FinReflectKG/snapshots/81819dfb4c95a96963c9a1848466cdf9612d47c0/data/train-00009-of-00103.parquet
ไฟล์มีอยู่: True
ขนาดไฟล์ (bytes): 16920015


## 6. ตรวจข้อมูลในไฟล์ก่อนกรอง

อ่าน metadata ของ Parquet เพื่อดูจำนวนแถวและชื่อคอลัมน์ โดยยังไม่ต้องแปลงทั้งไฟล์เป็น DataFrame


In [11]:
parquet_file = pq.ParquetFile(local_path)
print('จำนวนแถวก่อนกรอง:', parquet_file.metadata.num_rows)
print('จำนวน row groups:', parquet_file.metadata.num_row_groups)
print('คอลัมน์:', parquet_file.schema.names)


จำนวนแถวก่อนกรอง: 170033
จำนวน row groups: 171
คอลัมน์: ['triplet_id', 'entity', 'entity_type', 'relationship', 'target', 'target_type', 'start_date', 'end_date', 'extraction_type', 'ticker', 'year', 'source_file', 'page_id', 'chunk_id', 'chunk_text', 'triplet_length', 'chunk_text_length', 'has_context']


## 7. กรองแถวให้เหลือ ticker ในรายชื่อ SOX

filter_sox() รับ path เป็นข้อความ แล้วคืน pyarrow.Table ที่มีเฉพาะแถวที่ค่า ticker ตรงกับรายชื่อที่กำหนด


In [12]:
def filter_sox(local_path: str) -> pa.Table:
    return pq.read_table(
        local_path,
        filters=[('ticker', 'in', SOX_TICKERS)],
    )


In [13]:
table = filter_sox(local_path)
matched_tickers = sorted(set(table['ticker'].to_pylist())) if table.num_rows else []

print('ชนิดค่าที่คืนมา:', type(table).__name__)
print('จำนวนแถวหลังกรอง:', table.num_rows)
print('ticker ที่พบ:', matched_tickers)
print('คอลัมน์ใน Table:', table.column_names)
print('ชนิดข้อมูลแต่ละคอลัมน์:')
print(table.schema)


ชนิดค่าที่คืนมา: Table
จำนวนแถวหลังกรอง: 29014
ticker ที่พบ: ['ADI']
คอลัมน์ใน Table: ['triplet_id', 'entity', 'entity_type', 'relationship', 'target', 'target_type', 'start_date', 'end_date', 'extraction_type', 'ticker', 'year', 'source_file', 'page_id', 'chunk_id', 'chunk_text', 'triplet_length', 'chunk_text_length', 'has_context']
ชนิดข้อมูลแต่ละคอลัมน์:
triplet_id: string
entity: string
entity_type: string
relationship: string
target: string
target_type: string
start_date: string
end_date: string
extraction_type: string
ticker: string
year: int64
source_file: string
page_id: string
chunk_id: string
chunk_text: string
triplet_length: int64
chunk_text_length: int64
has_context: bool
-- schema metadata --
huggingface: '{"info": {"features": {"triplet_id": {"dtype": "string", "_' + 911


In [14]:
if table.num_rows:
    sample = table.slice(0, 1).to_pylist()[0]
    for key in ['triplet_id', 'entity', 'relationship', 'target', 'ticker', 'year', 'source_file', 'page_id', 'chunk_id']:
        print(f'{key}: {sample.get(key)!r}')
    print('chunk_text ตัวอย่าง:', str(sample.get('chunk_text', ''))[:220])
else:
    print('shard นี้ไม่มี ticker ที่ต้องการ ลองเปลี่ยน DEMO_SHARD_INDEX แล้วรันตั้งแต่ขั้นเลือก shard')


triplet_id: 'adi_2014_page_6_chunk_1_triplet_1'
entity: 'adi'
relationship: 'discloses'
target: 'revenue breakdown'
ticker: 'ADI'
year: 2014
source_file: 'ADI_10k_2014.pdf'
page_id: 'page_6'
chunk_id: 'chunk_1'
chunk_text ตัวอย่าง: ## Markets and Applications The breakdown of our fiscal 2014 revenue by end market is set out in the table below.


## 8. รวม Table ที่กรองได้

สคริปต์จริงวนหลาย shard และเก็บเฉพาะ Table ที่มีแถวไว้ใน sox_tables จากนั้น pa.concat_tables() รวมตารางเหล่านั้น ในเดโมนี้มีหนึ่ง shard จึงมีได้สูงสุดหนึ่ง Table


In [17]:
sox_tables = []
if table.num_rows > 0:
    sox_tables.append(table)

print('จำนวน Table ที่เก็บไว้:', len(sox_tables))

if sox_tables:
    combined = pa.concat_tables(sox_tables)
    print('ชนิด combined:', type(combined).__name__)
    print('จำนวนแถวใน combined:', combined.num_rows)
else:
    combined = None
    print('ยังไม่มีข้อมูลสำหรับรวม')


จำนวน Table ที่เก็บไว้: 1
ชนิด combined: Table
จำนวนแถวใน combined: 29014


## 9. เขียนผลเป็น Parquet

เซลล์นี้เขียนเฉพาะผลเดโมไปยัง OUTPUT_PATH ซึ่งแยกจากไฟล์ SOX หลัก


In [18]:
if combined is not None:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    pq.write_table(combined, OUTPUT_PATH)
    saved_file = pq.ParquetFile(OUTPUT_PATH)
    print('บันทึกที่:', OUTPUT_PATH)
    print('ขนาดไฟล์ (bytes):', OUTPUT_PATH.stat().st_size)
    print('จำนวนแถวในไฟล์:', saved_file.metadata.num_rows)
else:
    print('ไม่ได้เขียนไฟล์ เพราะไม่มีแถวที่ผ่านตัวกรอง')


บันทึกที่: /home/kantinan/.cache/semigraph/fetch_ds_notebook_demo/finreflectkg_sox_demo.parquet
ขนาดไฟล์ (bytes): 7438222
จำนวนแถวในไฟล์: 29014


## 10. สรุปจำนวนแถวต่อ ticker และปี

ส่วนนี้แสดงรูปแบบสรุปท้ายสคริปต์เดิม ผลจะนับแถว triple ไม่ใช่จำนวน Chunk ที่ไม่ซ้ำ


In [19]:
if combined is not None:
    df = combined.to_pandas()
    print(f'รวมทั้งหมด: {len(df):,} แถว')
    for ticker in SOX_TICKERS:
        subset = df[df['ticker'] == ticker]
        if len(subset):
            years = sorted(subset['year'].unique())
            print(f'{ticker}: {len(subset):,} triples, years={years}')
        else:
            print(f'{ticker}: ไม่พบใน shard นี้')
else:
    print('ไม่มีข้อมูลให้สรุป')


รวมทั้งหมด: 29,014 แถว
NVDA: ไม่พบใน shard นี้
AMD: ไม่พบใน shard นี้
INTC: ไม่พบใน shard นี้
AVGO: ไม่พบใน shard นี้
TXN: ไม่พบใน shard นี้
AMAT: ไม่พบใน shard นี้
QCOM: ไม่พบใน shard นี้
ADI: 29,014 triples, years=[np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
MU: ไม่พบใน shard นี้
MRVL: ไม่พบใน shard นี้
MCHP: ไม่พบใน shard นี้
KLAC: ไม่พบใน shard นี้
LRCX: ไม่พบใน shard นี้


## อ่านผลที่ได้อย่างไร

1. shards เป็นรายชื่อไฟล์บน Hugging Face
2. local_path เป็น path ของไฟล์ Parquet บนเครื่อง
3. table เป็นข้อมูลหลังกรอง ticker แล้ว
4. combined เป็น Table ที่รวมผลจาก shard ที่นำมาทดลอง
5. OUTPUT_PATH เป็นไฟล์ Parquet ที่บันทึกจากการทดลองนี้

Notebook นี้ประมวลผลเพียง shard เดียวเพื่อให้ดูผลแต่ละขั้นได้เร็ว ผลจึงไม่ใช่ corpus SOX ครบทุก shard และโค้ดชุดนี้ไม่ได้แบ่งข้อความเป็น Chunk ใหม่หรือสร้าง embedding
